# Correction : KNN avec Iris, sans normalisation

Objectif : utiliser KNN pour prédire l'espèce d'une fleur à partir du dataset Iris.

Dans cette correction, on ne normalise pas encore les données. Le but est de comprendre le mécanisme de base :

```text
nouvelle fleur -> distances -> k voisins -> vote -> prédiction
```

## 1. Importer les bibliothèques

In [ ]:
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

## 2. Charger le dataset Iris

`X` contient les 4 mesures des fleurs.

`y` contient l'espèce à prédire :

- `0` : setosa ;
- `1` : versicolor ;
- `2` : virginica.

In [ ]:
iris = load_iris()

X = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

y = pd.Series(iris.target)

X.head()

In [ ]:
iris.target_names

## 3. Séparer les données en train et test

- le train sert au modèle ;
- le test sert à vérifier le modèle sur des fleurs non vues ;
- `stratify=y` garde une répartition équilibrée des espèces.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

## 4. Créer et entraîner le modèle KNN

`n_neighbors=5` signifie que le modèle regarde les 5 fleurs les plus proches.

Cela ne veut pas dire qu'il utilise 5 variables : Iris contient bien 4 mesures par fleur.

In [ ]:
modele = KNeighborsClassifier(n_neighbors=5)
modele.fit(X_train, y_train)

Avec KNN, entraîner signifie surtout mémoriser les données d'entraînement.

Le vrai calcul arrive au moment de la prédiction : le modèle compare la nouvelle fleur aux fleurs du train.

## 5. Évaluer le modèle

In [ ]:
score_test = modele.score(X_test, y_test)

print("Score sur les données de test :", score_test)

Le score indique la proportion de fleurs bien classées dans le jeu de test.

## 6. Prendre une fleur inconnue

On choisit une fleur du test. Elle est inconnue pour le modèle, car elle n'a pas été utilisée pendant `.fit(...)`.

In [ ]:
index = X_test.index[0]

nouvelle_fleur = X_test.loc[[index]]
vraie_classe = y_test.loc[index]

nouvelle_fleur

## 7. Prédire son espèce

In [ ]:
prediction = modele.predict(nouvelle_fleur)

print("Vraie espèce   :", iris.target_names[vraie_classe])
print("Espèce prédite :", iris.target_names[prediction[0]])

## 8. Observer les 5 plus proches voisines

`kneighbors` permet de voir concrètement quelles fleurs ont été utilisées pour décider.

In [ ]:
distances, indices = modele.kneighbors(nouvelle_fleur)

voisins = X_train.iloc[indices[0]].copy()
voisins["distance"] = distances[0]
voisins["classe"] = y_train.iloc[indices[0]].values
voisins["espece"] = [
    iris.target_names[classe]
    for classe in voisins["classe"]
]

voisins

KNN a comparé la nouvelle fleur à toutes les fleurs du train, puis il a gardé les 5 distances les plus petites.

## 9. Comprendre le vote

In [ ]:
vote = voisins["espece"].value_counts()

vote

La classe prédite est celle qui apparaît le plus souvent parmi les 5 voisines.

Résumé :

```text
distances -> 5 plus proches voisines -> vote -> espèce prédite
```